# R-CNN Обнаружение и Классификация Объектов (Улучшенная версия)

## 🚀 Улучшения:
- ✅ **Автоматическая проверка датасета** - определяет количество классов и распределение
- ✅ **Предупреждения о проблемах** - если датасет неправильно распределен
- ✅ **Русские графики с подсказками** - понятная визуализация с интерпретацией
- ✅ **Подробные комментарии** - каждая строка кода объяснена
- ✅ **Автонастройка параметров** - NUM_CLASSES определяется автоматически

## 📋 Что делает этот ноутбук:
- Загружает данные из Google Drive
- Проверяет корректность датасета
- Обучает модель R-CNN для обнаружения объектов
- Визуализирует результаты с подсказками
- Сохраняет обученную модель

## 1. Установка необходимых библиотек

In [ ]:
# Установка библиотек для работы с нейронными сетями и визуализацией
!pip install torch torchvision  # PyTorch - основная библиотека для глубокого обучения
!pip install matplotlib pillow numpy  # Визуализация и работа с изображениями
!pip install albumentations  # Аугментация данных (дополнительные трансформации)
!pip install tqdm  # Прогресс-бары для отслеживания процесса обучения

## 2. Импорт библиотек

In [ ]:
# ============================================
# ИМПОРТ БИБЛИОТЕК
# ============================================

# Библиотеки для работы с PyTorch
import torch  # Основная библиотека PyTorch
import torch.nn as nn  # Модуль для создания нейронных сетей
import torchvision  # Библиотека для компьютерного зрения
from torchvision.models.detection import fasterrcnn_resnet50_fpn  # Предобученная модель Faster R-CNN
from torchvision.models.detection.faster_rcnn import FastRCNNPredictor  # Классификатор для R-CNN
from torch.utils.data import Dataset, DataLoader  # Для работы с датасетами

# Библиотеки для работы с данными и визуализацией
import numpy as np  # Работа с массивами и математические операции
import matplotlib.pyplot as plt  # Построение графиков
import matplotlib.patches as patches  # Рисование прямоугольников на изображениях
from matplotlib import font_manager  # Управление шрифтами для русских букв
from PIL import Image  # Работа с изображениями
import os  # Работа с файловой системой
import glob  # Поиск файлов по шаблону
from tqdm import tqdm  # Прогресс-бары
import json  # Сохранение/загрузка данных в формате JSON
import random  # Генерация случайных чисел
from collections import defaultdict  # Словари с значениями по умолчанию
import warnings  # Управление предупреждениями

# ============================================
# НАСТРОЙКА ОКРУЖЕНИЯ
# ============================================

# Установка seed для воспроизводимости результатов
# Это гарантирует, что при повторном запуске результаты будут одинаковыми
RANDOM_SEED = 42
random.seed(RANDOM_SEED)
np.random.seed(RANDOM_SEED)
torch.manual_seed(RANDOM_SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed(RANDOM_SEED)
    torch.cuda.manual_seed_all(RANDOM_SEED)  # Для multi-GPU
    # Для максимальной воспроизводимости (может немного замедлить обучение)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

# Настройка matplotlib для отображения русского текста
plt.rcParams['font.family'] = 'DejaVu Sans'  # Шрифт с поддержкой кириллицы
plt.rcParams['axes.unicode_minus'] = False  # Корректное отображение минуса

# Проверка доступности GPU
device = torch.device('cuda') if torch.cuda.is_available() else torch.device('cpu')
print('='*60)
print(f'🖥️  Используемое устройство: {device}')
if torch.cuda.is_available():
    print(f'📊 GPU: {torch.cuda.get_device_name(0)}')
    print(f'💾 Доступная память GPU: {torch.cuda.get_device_properties(0).total_memory / 1024**3:.2f} ГБ')
else:
    print('⚠️  GPU не обнаружен, используется CPU (обучение будет медленнее)')
print('='*60)

## 3. Подключение Google Drive

In [ ]:
# ============================================
# ПОДКЛЮЧЕНИЕ GOOGLE DRIVE
# ============================================

from google.colab import drive
drive.mount('/content/drive')

# ============================================
# НАСТРОЙКА ПУТЕЙ К ДАННЫМ
# ============================================

# ⚠️ ВАЖНО: Измените этот путь на ваш путь к датасету в Google Drive
DATASET_PATH = '/content/drive/MyDrive/dataset'  # ← ИЗМЕНИТЕ ЭТОТ ПУТЬ!

# Структура датасета должна быть следующей:
# dataset/
#   ├── train/
#   │   ├── images/  (файлы *.png)
#   │   └── labels/  (файлы *.txt в формате YOLO)
#   ├── val/
#   │   ├── images/
#   │   └── labels/
#   └── test/
#       ├── images/
#       └── labels/

# Автоматическое формирование путей к подпапкам
TRAIN_IMAGES = os.path.join(DATASET_PATH, 'train/images')
TRAIN_LABELS = os.path.join(DATASET_PATH, 'train/labels')
VAL_IMAGES = os.path.join(DATASET_PATH, 'val/images')
VAL_LABELS = os.path.join(DATASET_PATH, 'val/labels')
TEST_IMAGES = os.path.join(DATASET_PATH, 'test/images')
TEST_LABELS = os.path.join(DATASET_PATH, 'test/labels')

# Путь для сохранения результатов (модель, графики, метрики)
OUTPUT_PATH = '/content/drive/MyDrive/rcnn_output'
os.makedirs(OUTPUT_PATH, exist_ok=True)  # Создаем папку, если её нет

print('\n' + '='*60)
print('✅ Google Drive успешно подключен!')
print('='*60)
print(f'📁 Путь к датасету: {DATASET_PATH}')
print(f'💾 Путь для сохранения результатов: {OUTPUT_PATH}')
print('='*60)

## 4. Автоматическая проверка и диагностика датасета

Эта ячейка проверяет:
- ✅ Существование файлов
- ✅ Количество классов
- ✅ Распределение классов по выборкам
- ✅ Процентное соотношение train/val/test
- ⚠️ Предупреждает о потенциальных проблемах

In [ ]:
# ============================================
# ФУНКЦИИ ДЛЯ ДИАГНОСТИКИ ДАТАСЕТА
# ============================================

def check_dataset_classes(labels_dir):
    """
    Проверяет какие классы присутствуют в датасете.
    
    Args:
        labels_dir: путь к папке с аннотациями (.txt файлы)
    
    Returns:
        all_classes: отсортированный список уникальных классов
    """
    label_files = glob.glob(os.path.join(labels_dir, '*.txt'))
    all_classes = set()  # Используем set для хранения уникальных значений
    
    for label_file in label_files:
        with open(label_file, 'r') as f:
            for line in f.readlines():
                parts = line.strip().split()
                if len(parts) >= 5:  # Формат YOLO: class x y w h
                    class_id = int(parts[0])
                    all_classes.add(class_id)
    
    return sorted(all_classes)


def count_images(images_dir):
    """
    Подсчитывает количество изображений в папке.
    
    Args:
        images_dir: путь к папке с изображениями
    
    Returns:
        count: количество .png файлов
    """
    return len(glob.glob(os.path.join(images_dir, '*.png')))


# ============================================
# ПРОВЕРКА ДАТАСЕТА
# ============================================

print('\n' + '='*60)
print('🔍 ДИАГНОСТИКА ДАТАСЕТА')
print('='*60)

# Подсчет количества изображений в каждой выборке
train_count = count_images(TRAIN_IMAGES)
val_count = count_images(VAL_IMAGES)
test_count = count_images(TEST_IMAGES)
total_count = train_count + val_count + test_count

print(f'\n📊 Количество изображений:')
print(f'  Train: {train_count} ({train_count/total_count*100:.1f}%)')
print(f'  Val:   {val_count} ({val_count/total_count*100:.1f}%)')
print(f'  Test:  {test_count} ({test_count/total_count*100:.1f}%)')
print(f'  Всего: {total_count}')

# Проверка классов в каждой выборке
print(f'\n🏷️  Анализ классов:')
print('-'*60)

train_classes = check_dataset_classes(TRAIN_LABELS)
val_classes = check_dataset_classes(VAL_LABELS)
test_classes = check_dataset_classes(TEST_LABELS)

print(f'Train классы: {train_classes}')
print(f'Val классы:   {val_classes}')
print(f'Test классы:  {test_classes}')

# Объединение всех классов
all_classes = sorted(set(train_classes + val_classes + test_classes))
print(f'\nВсе уникальные классы: {all_classes}')
print(f'Количество уникальных классов: {len(all_classes)}')

# ============================================
# АВТОМАТИЧЕСКОЕ ОПРЕДЕЛЕНИЕ NUM_CLASSES
# ============================================

# NUM_CLASSES = количество классов объектов + 1 (для фона)
# Например: если у вас классы [0, 1, 2], то max=2, NUM_CLASSES=2+1+1=4
if len(all_classes) > 0:
    NUM_CLASSES = max(all_classes) + 2  # +1 для индексации с 0, +1 для фона
else:
    NUM_CLASSES = 2  # Минимальное значение (фон + 1 класс)
    
print(f'\n✅ Автоматически определено NUM_CLASSES = {NUM_CLASSES}')
print(f'   (максимальный class_id: {max(all_classes) if all_classes else 0} + 1 для фона + 1 для индексации)')

# ============================================
# ПРОВЕРКА ПРОБЛЕМ В ДАТАСЕТЕ
# ============================================

print(f'\n⚠️  ПРОВЕРКА ВОЗМОЖНЫХ ПРОБЛЕМ:')
print('-'*60)

has_issues = False

# Проверка 1: Минимальное количество данных
if total_count < 50:
    print(f'⚠️  ПРЕДУПРЕЖДЕНИЕ: Мало данных ({total_count} изображений)')
    print(f'   Рекомендуется минимум 100 изображений для хорошего обучения')
    has_issues = True

# Проверка 2: Распределение train/val/test
train_ratio = train_count / total_count if total_count > 0 else 0
val_ratio = val_count / total_count if total_count > 0 else 0
test_ratio = test_count / total_count if total_count > 0 else 0

# Рекомендуемое распределение: 70% train, 15% val, 15% test
if train_ratio < 0.6:
    print(f'⚠️  ПРЕДУПРЕЖДЕНИЕ: Мало данных для обучения ({train_ratio*100:.1f}%)')
    print(f'   Рекомендуется минимум 60-70% для train')
    has_issues = True

if val_ratio < 0.1 or val_ratio > 0.3:
    print(f'⚠️  ПРЕДУПРЕЖДЕНИЕ: Нестандартное распределение val ({val_ratio*100:.1f}%)')
    print(f'   Рекомендуется 10-20% для val')
    has_issues = True

# Проверка 3: Наличие всех классов во всех выборках
missing_in_train = set(all_classes) - set(train_classes)
missing_in_val = set(all_classes) - set(val_classes)
missing_in_test = set(all_classes) - set(test_classes)

if missing_in_train:
    print(f'🚨 КРИТИЧЕСКАЯ ПРОБЛЕМА: В train отсутствуют классы {sorted(missing_in_train)}')
    print(f'   Модель НЕ СМОЖЕТ обучиться на этих классах!')
    has_issues = True

if missing_in_val:
    print(f'🚨 КРИТИЧЕСКАЯ ПРОБЛЕМА: В val отсутствуют классы {sorted(missing_in_val)}')
    print(f'   Валидация будет неполной!')
    has_issues = True

if missing_in_test:
    print(f'⚠️  ПРЕДУПРЕЖДЕНИЕ: В test отсутствуют классы {sorted(missing_in_test)}')
    print(f'   Тестирование будет неполным!')
    has_issues = True

# Проверка 4: Классы только в одной выборке
only_in_val = set(val_classes) - set(train_classes)
only_in_test = set(test_classes) - set(train_classes)

if only_in_val:
    print(f'🚨 КРИТИЧЕСКАЯ ПРОБЛЕМА: Классы {sorted(only_in_val)} есть только в val, но отсутствуют в train!')
    print(f'   Модель НИКОГДА не видела эти классы при обучении!')
    print(f'   ➡️  РЕШЕНИЕ: Переместите часть изображений из val в train')
    has_issues = True

if only_in_test:
    print(f'🚨 КРИТИЧЕСКАЯ ПРОБЛЕМА: Классы {sorted(only_in_test)} есть только в test, но отсутствуют в train!')
    print(f'   Модель НИКОГДА не видела эти классы при обучении!')
    print(f'   ➡️  РЕШЕНИЕ: Переместите часть изображений из test в train')
    has_issues = True

if not has_issues:
    print('✅ Проблем не обнаружено! Датасет готов к использованию.')
else:
    print('\n❗ Обнаружены проблемы с датасетом!')
    print('   Рекомендуется исправить проблемы перед обучением.')
    print('   Но вы можете продолжить для экспериментов.')

print('='*60)

## 5. Класс Dataset для работы с данными в формате YOLO

In [ ]:
# ============================================
# КЛАСС ДЛЯ ЗАГРУЗКИ ДАТАСЕТА
# ============================================

class ObjectDetectionDataset(Dataset):
    """
    Датасет для обнаружения объектов.
    
    Загружает изображения (.png) и аннотации в формате YOLO (.txt).
    
    Формат аннотаций YOLO (каждая строка - один объект):
    class_id x_center y_center width height
    
    Где:
    - class_id: номер класса (0, 1, 2, ...)
    - x_center: X координата центра объекта (0.0 - 1.0)
    - y_center: Y координата центра объекта (0.0 - 1.0)
    - width: ширина объекта (0.0 - 1.0)
    - height: высота объекта (0.0 - 1.0)
    
    Все координаты нормализованы относительно размера изображения.
    """
    
    def __init__(self, images_dir, labels_dir, transforms=None):
        """
        Инициализация датасета.
        
        Args:
            images_dir: путь к папке с изображениями (.png файлы)
            labels_dir: путь к папке с аннотациями (.txt файлы)
            transforms: трансформации для аугментации данных (необязательно)
        """
        self.images_dir = images_dir
        self.labels_dir = labels_dir
        self.transforms = transforms
        
        # Получение списка всех изображений в папке
        self.image_files = sorted(glob.glob(os.path.join(images_dir, '*.png')))
        
        # Проверка, что изображения найдены
        if len(self.image_files) == 0:
            raise ValueError(f'❌ Изображения не найдены в {images_dir}')
            
        print(f'✅ Загружено {len(self.image_files)} изображений из {images_dir}')
    
    def __len__(self):
        """Возвращает количество изображений в датасете."""
        return len(self.image_files)
    
    def __getitem__(self, idx):
        """
        Загружает одно изображение и его аннотации.
        
        Args:
            idx: индекс изображения
        
        Returns:
            image: тензор изображения [3, H, W]
            target: словарь с полями:
                - boxes: координаты объектов [N, 4] в формате [x_min, y_min, x_max, y_max]
                - labels: метки классов [N]
                - image_id: идентификатор изображения
        """
        # ========================================
        # ЗАГРУЗКА ИЗОБРАЖЕНИЯ
        # ========================================
        img_path = self.image_files[idx]
        image = Image.open(img_path).convert('RGB')  # Конвертируем в RGB на всякий случай
        img_width, img_height = image.size  # Получаем размеры изображения
        
        # ========================================
        # ЗАГРУЗКА АННОТАЦИЙ
        # ========================================
        
        # Формируем путь к файлу с аннотациями
        # Например: image1.png → image1.txt
        label_filename = os.path.basename(img_path).replace('.png', '.txt')
        label_path = os.path.join(self.labels_dir, label_filename)
        
        boxes = []  # Список координат объектов
        labels = []  # Список меток классов
        
        # Чтение аннотаций, если файл существует
        if os.path.exists(label_path):
            with open(label_path, 'r') as f:
                for line in f.readlines():
                    # Парсинг строки: class_id x_center y_center width height
                    parts = line.strip().split()
                    
                    if len(parts) == 5:
                        # Извлечение значений
                        class_id = int(parts[0])  # Класс объекта
                        x_center = float(parts[1])  # Центр X (нормализованный)
                        y_center = float(parts[2])  # Центр Y (нормализованный)
                        width = float(parts[3])  # Ширина (нормализованная)
                        height = float(parts[4])  # Высота (нормализованная)
                        
                        # ========================================
                        # КОНВЕРТАЦИЯ ИЗ ФОРМАТА YOLO В R-CNN
                        # ========================================
                        
                        # YOLO формат: (x_center, y_center, width, height) - нормализованные
                        # R-CNN формат: (x_min, y_min, x_max, y_max) - абсолютные пиксели
                        
                        # Вычисление координат углов в пикселях
                        x_min = (x_center - width / 2) * img_width  # Левый край
                        y_min = (y_center - height / 2) * img_height  # Верхний край
                        x_max = (x_center + width / 2) * img_width  # Правый край
                        y_max = (y_center + height / 2) * img_height  # Нижний край
                        
                        # Проверка корректности координат (не выходят за границы изображения)
                        x_min = max(0, min(x_min, img_width))
                        y_min = max(0, min(y_min, img_height))
                        x_max = max(0, min(x_max, img_width))
                        y_max = max(0, min(y_max, img_height))
                        
                        # Проверка, что бокс имеет положительную площадь
                        if x_max > x_min and y_max > y_min:
                            boxes.append([x_min, y_min, x_max, y_max])
                            # +1 потому что в R-CNN класс 0 зарезервирован для фона
                            labels.append(class_id + 1)
        
        # ========================================
        # СОЗДАНИЕ ТЕНЗОРОВ
        # ========================================
        
        # Если нет объектов на изображении, создаем пустые тензоры
        if len(boxes) == 0:
            boxes = torch.zeros((0, 4), dtype=torch.float32)
            labels = torch.zeros((0,), dtype=torch.int64)
        else:
            # Конвертируем списки в тензоры PyTorch
            boxes = torch.as_tensor(boxes, dtype=torch.float32)
            labels = torch.as_tensor(labels, dtype=torch.int64)
        
        # Преобразование изображения в тензор [3, H, W]
        image = torchvision.transforms.functional.to_tensor(image)
        
        # ========================================
        # ФОРМИРОВАНИЕ ВЫХОДНЫХ ДАННЫХ
        # ========================================
        
        # Создание словаря с целевыми данными для модели
        target = {
            'boxes': boxes,  # Координаты объектов
            'labels': labels,  # Метки классов
            'image_id': torch.tensor([idx])  # ID изображения
        }
        
        return image, target

## 6. Создание датасетов и загрузчиков данных

In [ ]:
# ============================================
# ГИПЕРПАРАМЕТРЫ ЗАГРУЗКИ ДАННЫХ
# ============================================

# Размер батча - сколько изображений обрабатывается за одну итерацию
# ⚠️ Если возникает ошибка Out of Memory (OOM), уменьшите это значение до 2 или 1
BATCH_SIZE = 4

# Количество потоков для загрузки данных
# Ускоряет загрузку, но использует больше памяти
NUM_WORKERS = 2

# NUM_CLASSES уже определен автоматически в предыдущей ячейке!
print(f'\n📊 Параметры загрузки данных:')
print(f'  BATCH_SIZE: {BATCH_SIZE}')
print(f'  NUM_WORKERS: {NUM_WORKERS}')
print(f'  NUM_CLASSES: {NUM_CLASSES} (определено автоматически)')

# ============================================
# СОЗДАНИЕ ДАТАСЕТОВ
# ============================================

print(f'\n🔄 Создание датасетов...')

# Создаем датасеты для каждой выборки
train_dataset = ObjectDetectionDataset(TRAIN_IMAGES, TRAIN_LABELS)
val_dataset = ObjectDetectionDataset(VAL_IMAGES, VAL_LABELS)
test_dataset = ObjectDetectionDataset(TEST_IMAGES, TEST_LABELS)

# ============================================
# ФУНКЦИЯ COLLATE ДЛЯ БАТЧЕЙ
# ============================================

def collate_fn(batch):
    """
    Пользовательская функция для объединения элементов в батч.
    
    Необходима потому что:
    - У разных изображений может быть разное количество объектов
    - Стандартный collate_fn PyTorch не умеет работать с такими данными
    
    Args:
        batch: список из (image, target) пар
    
    Returns:
        Два кортежа: (images), (targets)
    """
    return tuple(zip(*batch))

# ============================================
# СОЗДАНИЕ ЗАГРУЗЧИКОВ ДАННЫХ (DataLoader)
# ============================================

# DataLoader автоматизирует:
# - Формирование батчей
# - Перемешивание данных (shuffle)
# - Многопоточную загрузку (num_workers)

# Загрузчик для обучающей выборки
train_loader = DataLoader(
    train_dataset,
    batch_size=BATCH_SIZE,
    shuffle=True,  # Перемешиваем данные каждую эпоху для лучшего обучения
    num_workers=NUM_WORKERS,
    collate_fn=collate_fn
)

# Загрузчик для валидационной выборки
val_loader = DataLoader(
    val_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,  # Не перемешиваем, порядок не важен
    num_workers=NUM_WORKERS,
    collate_fn=collate_fn
)

# Загрузчик для тестовой выборки
test_loader = DataLoader(
    test_dataset,
    batch_size=1,  # По одному изображению для точной оценки
    shuffle=False,
    num_workers=NUM_WORKERS,
    collate_fn=collate_fn
)

print(f'\n✅ Загрузчики данных созданы:')
print(f'  Обучающая выборка: {len(train_loader)} батчей по {BATCH_SIZE} изображений')
print(f'  Валидационная выборка: {len(val_loader)} батчей по {BATCH_SIZE} изображений')
print(f'  Тестовая выборка: {len(test_loader)} батчей по 1 изображению')

## 7. Создание модели R-CNN (Faster R-CNN)

### Что такое Faster R-CNN?

**Faster R-CNN** - это продвинутая архитектура для обнаружения объектов, которая работает в два этапа:

1. **Region Proposal Network (RPN)** - предлагает области, где могут находиться объекты
2. **Fast R-CNN** - классифицирует объекты в этих областях и уточняет координаты

**Преимущества:**
- ✅ Высокая точность обнаружения
- ✅ Предобученный backbone (ResNet50) на ImageNet
- ✅ Feature Pyramid Network (FPN) для обнаружения объектов разных размеров

In [ ]:
# ============================================
# ОЧИСТКА ПАМЯТИ GPU ПЕРЕД СОЗДАНИЕМ МОДЕЛИ
# ============================================

import gc  # Garbage collector для очистки памяти

# Принудительная очистка неиспользуемых объектов
gc.collect()

# Очистка кеша CUDA (если используется GPU)
if torch.cuda.is_available():
    torch.cuda.empty_cache()
    print('✅ Кеш GPU очищен')

# ============================================
# ФУНКЦИЯ СОЗДАНИЯ МОДЕЛИ
# ============================================

def create_model(num_classes):
    """
    Создание модели Faster R-CNN с предобученным backbone ResNet50-FPN.
    
    Архитектура:
    1. ResNet50 - извлечение признаков из изображения (backbone)
    2. FPN (Feature Pyramid Network) - работа с объектами разных размеров
    3. RPN (Region Proposal Network) - генерация предложений регионов
    4. ROI Head - классификация и регрессия bounding box
    
    Args:
        num_classes: количество классов (включая фон)
                    Например: фон + 2 класса объектов = 3
    
    Returns:
        model: модель Faster R-CNN готовая к обучению
    """
    
    # Загрузка предобученной модели Faster R-CNN
    # Веса обучены на датасете COCO (80 классов объектов)
    # weights='DEFAULT' загружает последние рекомендуемые веса
    model = fasterrcnn_resnet50_fpn(
        weights=torchvision.models.detection.FasterRCNN_ResNet50_FPN_Weights.DEFAULT
    )
    
    # ========================================
    # ЗАМЕНА ГОЛОВЫ КЛАССИФИКАЦИИ
    # ========================================
    
    # Получаем количество входных признаков для классификатора
    # Это зависит от архитектуры backbone (для ResNet50 = 1024)
    in_features = model.roi_heads.box_predictor.cls_score.in_features
    
    # Заменяем голову классификации на новую для нашего количества классов
    # Это позволяет модели учиться классифицировать наши объекты
    model.roi_heads.box_predictor = FastRCNNPredictor(in_features, num_classes)
    
    return model

# ============================================
# СОЗДАНИЕ И ИНИЦИАЛИЗАЦИЯ МОДЕЛИ
# ============================================

print(f'\n🏗️  Создание модели Faster R-CNN...')

# Создаем модель с автоматически определенным количеством классов
model = create_model(NUM_CLASSES)

# Перемещаем модель на GPU (если доступен) или CPU
model.to(device)

print(f'\n✅ Модель Faster R-CNN успешно создана!')
print(f'📊 Параметры модели:')
print(f'  Количество классов: {NUM_CLASSES}')
print(f'  Устройство: {device}')

# ============================================
# ПОДСЧЕТ ПАРАМЕТРОВ МОДЕЛИ
# ============================================

# Общее количество параметров (всех слоев)
total_params = sum(p.numel() for p in model.parameters())

# Количество обучаемых параметров (с requires_grad=True)
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)

print(f'\n📈 Статистика параметров:')
print(f'  Всего параметров: {total_params:,}')
print(f'  Обучаемых параметров: {trainable_params:,}')
print(f'  Замороженных параметров: {total_params - trainable_params:,}')

## 8. Настройка оптимизатора и планировщика

### Что такое оптимизатор?
**Оптимизатор** - алгоритм, который обновляет веса модели для минимизации ошибки.

Мы используем **SGD (Stochastic Gradient Descent)** с моментом:
- ✅ Стабильное обучение
- ✅ Хорошо работает с R-CNN
- ✅ Moment помогает избежать застревания в локальных минимумах

### Что такое планировщик?
**Learning Rate Scheduler** постепенно уменьшает скорость обучения:
- Вначале - быстрое обучение с большим шагом
- Потом - тонкая настройка с маленьким шагом

In [ ]:
# ============================================
# ГИПЕРПАРАМЕТРЫ ОБУЧЕНИЯ
# ============================================

# Количество эпох - сколько раз модель пройдет по всему датасету
# ⚠️ Для лучших результатов увеличьте до 50-100 эпох
NUM_EPOCHS = 20

# Скорость обучения (learning rate)
# Определяет размер шага при обновлении весов
# 💡 Больше = быстрее обучение, но может пропустить оптимум
# 💡 Меньше = медленнее обучение, но точнее находит оптимум
LEARNING_RATE = 0.005

# Момент (momentum) для SGD
# Помогает ускорить сходимость и избежать застревания
# 💡 Типичные значения: 0.9 - 0.99
MOMENTUM = 0.9

# L2 регуляризация (weight decay)
# Предотвращает переобучение, штрафуя большие веса
# 💡 Меньше значение = меньше регуляризации
WEIGHT_DECAY = 0.0005

print('='*60)
print('⚙️  ПАРАМЕТРЫ ОБУЧЕНИЯ')
print('='*60)
print(f'Количество эпох: {NUM_EPOCHS}')
print(f'Скорость обучения: {LEARNING_RATE}')
print(f'Момент: {MOMENTUM}')
print(f'Weight decay: {WEIGHT_DECAY}')

# ============================================
# СОЗДАНИЕ ОПТИМИЗАТОРА
# ============================================

# Получаем только обучаемые параметры (с requires_grad=True)
params = [p for p in model.parameters() if p.requires_grad]

# Создаем оптимизатор SGD (Stochastic Gradient Descent)
optimizer = torch.optim.SGD(
    params,  # Параметры для оптимизации
    lr=LEARNING_RATE,  # Скорость обучения
    momentum=MOMENTUM,  # Момент для ускорения
    weight_decay=WEIGHT_DECAY  # L2 регуляризация
)

# ============================================
# СОЗДАНИЕ ПЛАНИРОВЩИКА СКОРОСТИ ОБУЧЕНИЯ
# ============================================

# StepLR уменьшает learning rate каждые step_size эпох в gamma раз
# Например: lr=0.005 → 0.0005 → 0.00005
lr_scheduler = torch.optim.lr_scheduler.StepLR(
    optimizer,
    step_size=3,  # Уменьшать каждые 3 эпохи
    gamma=0.1  # Уменьшать в 10 раз
)

print(f'\n✅ Оптимизатор и планировщик настроены!')
print(f'\n📉 График изменения learning rate:')
print(f'  Эпохи 1-3:   lr = {LEARNING_RATE:.6f}')
print(f'  Эпохи 4-6:   lr = {LEARNING_RATE * 0.1:.6f}')
print(f'  Эпохи 7-9:   lr = {LEARNING_RATE * 0.01:.6f}')
print(f'  Эпохи 10+:   lr = {LEARNING_RATE * 0.001:.6f}')
print('='*60)